# GẮN NHÃN NGHỀ: gắn nghề vào nhóm

=> Tóm lại thì sẽ cần Chuẩn bị dữ liệu nghề: đã có danh sách nghề + biết mỗi nghề đi với bao nhiêu skill

Thay vì để một đống nghề nằm lộn xộn, giờ mình bắt đầu chia chúng vào các nhóm như:

- nhóm Frontend

- nhóm Backend

- nhóm Data/AI

- nhóm DevOps/Cloud

- nhóm QA/Test

- nhóm BA/PM
...

Đấy, taxonomy ở đây  cứ hiểu đơn giản là:

    cây phân loại nghề

Không cần nghĩ cao siêu hơn.

Trong file 09 sẽ có gì?

Mình chỉ cần thêm 2 cột mới vào file occupation hiện tại:

- group → cái này  đã có rồi, là core / extended

- taxonomy_group → nhóm nghề chính

- taxonomy_subgroup → nhóm nghề con

* Ví dụ như này: 

| occupation           | group    | skill_count | taxonomy_group         | taxonomy_subgroup  |
| -------------------- | -------- | ----------: | ---------------------- | ------------------ |
| front-end developer  | core     |          35 | Software Development   | Frontend           |
| back-end developer   | core     |          42 | Software Development   | Backend            |
| full-stack developer | core     |          50 | Software Development   | Fullstack          |
| data scientist       | core     |          38 | Data & AI              | Data Science       |
| devops engineer      | extended |          28 | Infrastructure & Cloud | DevOps             |
| ICT project manager  | extended |          18 | Product / Delivery     | Project Management |


Ở bước này, dữ liệu được lấy từ file 06 và file 08.

- File 06 chứa danh sách occupation tổng, tức là danh sách các nghề đã được giữ lại sau quá trình lọc.

- File 08 chứa số lượng skill liên quan tới từng occupation.

Sau đó, hai file này được ghép lại với nhau để mỗi occupation vừa có:

- tên nghề

- nhóm core/extended

- số lượng skill tương ứng

Tiếp theo, thêm 2 cột mới:

- taxonomy_group: nhóm nghề lớn

- taxonomy_subgroup: nhóm nghề nhỏ hơn nằm bên trong nhóm lớn

Ví dụ:

- Software Development là nhóm lớn

- Frontend là nhóm nhỏ bên trong đó

Ở bước này, taxonomy mới chỉ dừng ở mức nhóm lớn và nhóm nhỏ, chưa đi sâu xuống nhiều tầng hơn.

Cách xử lý trong code

Sau khi xác định file đầu vào, chương trình sẽ:

1. Đọc dữ liệu từ file 06 và file 08

2. Chuẩn hóa text để tránh lỗi khi ghép dữ liệu, ví dụ:

- chuyển về chữ thường

- bỏ dấu nếu có

- bỏ bớt ký tự đặc biệt

3. Xác định các cột quan trọng trong mỗi file

4. Ghép dữ liệu của hai file lại với nhau

5. Gán nhóm nghề thủ công bằng rule từ khóa

    - Nếu occupation không khớp rule nào thì tạm gán vào:

    - Other IT
    
    - Unclassified

6. Cuối cùng, dữ liệu được sắp xếp lại cho dễ nhìn và xuất ra file mới



# 1. CẤU HÌNH ĐƯỜNG DẪN FILE

In [ ]:
import pandas as pd
import re
import unicodedata
from pathlib import Path

FILE_06 = "/Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/notebook/ESCO_taxonomy/notebook_clean/06_esco_it_all_occupations.csv"
FILE_08 = "/Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/notebook/ESCO_taxonomy/notebook_clean/08_esco_occupation_skill_count.csv"
OUTPUT_FILE = "/Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/notebook/ESCO_taxonomy/notebook_clean/09_occupation_taxonomy_seed.csv"

# 2. HÀM ĐỌC FILE

In [4]:
def read_table(file_path: str) -> pd.DataFrame:
    path = Path(file_path)
    if not path.exists():
        raise FileNotFoundError(f"Không tìm thấy file: {file_path}")

    if path.suffix.lower() in [".xlsx", ".xls"]:
        return pd.read_excel(path)
    elif path.suffix.lower() == ".csv":
        return pd.read_csv(path)
    else:
        raise ValueError(f"Định dạng file chưa hỗ trợ: {path.suffix}")

# 3. CHUẨN HÓA TEXT

In [5]:
def normalize_text(text):
    if pd.isna(text):
        return ""
    text = str(text).strip().lower()

    # bỏ dấu nếu có
    text = unicodedata.normalize("NFKD", text)
    text = "".join(c for c in text if not unicodedata.combining(c))

    # thay ký tự đặc biệt thành khoảng trắng
    text = re.sub(r"[^a-z0-9+#./\- ]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

# 4. TÌM TÊN CỘT PHÙ HỢP : để đỡ phải sửa code quá nhiều nếu tên cột lệch nhẹ (xây dựng hàm tìm kiếm tên cột)

In [6]:
def find_column(df: pd.DataFrame, candidates: list[str], required=True):
    cols_map = {normalize_text(c): c for c in df.columns}

    for cand in candidates:
        cand_norm = normalize_text(cand)
        if cand_norm in cols_map:
            return cols_map[cand_norm]

    # tìm gần đúng
    for col in df.columns:
        col_norm = normalize_text(col)
        for cand in candidates:
            cand_norm = normalize_text(cand)
            if cand_norm in col_norm or col_norm in cand_norm:
                return col

    if required:
        raise KeyError(
            f"Không tìm thấy cột phù hợp. Candidates={candidates}. "
            f"Các cột hiện có: {list(df.columns)}"
        )
    return None

# 5. RULE GÁN TAXONOMY

In [7]:
RULES = [
    # Software Development - Frontend
    {
        "keywords": [
            "front end", "frontend", "front-end",
            "ui developer", "web ui", "react", "vue", "angular"
        ],
        "taxonomy_group": "Software Development",
        "taxonomy_subgroup": "Frontend"
    },

    # Software Development - Backend
    {
        "keywords": [
            "back end", "backend", "back-end",
            "server side", "api developer", "java developer",
            ".net developer", "php developer", "nodejs developer",
            "python developer"
        ],
        "taxonomy_group": "Software Development",
        "taxonomy_subgroup": "Backend"
    },

    # Software Development - Fullstack
    {
        "keywords": [
            "full stack", "fullstack", "full-stack"
        ],
        "taxonomy_group": "Software Development",
        "taxonomy_subgroup": "Fullstack"
    },

    # Software Development - Mobile
    {
        "keywords": [
            "mobile developer", "android developer", "ios developer",
            "flutter developer", "react native", "swift developer", "kotlin developer"
        ],
        "taxonomy_group": "Software Development",
        "taxonomy_subgroup": "Mobile"
    },

    # Software Development - General
    {
        "keywords": [
            "software developer", "software engineer", "application developer",
            "programmer", "web developer"
        ],
        "taxonomy_group": "Software Development",
        "taxonomy_subgroup": "General Software Development"
    },

    # Data & AI - Data Analysis
    {
        "keywords": [
            "data analyst", "business intelligence", "bi analyst", "analytics specialist"
        ],
        "taxonomy_group": "Data & AI",
        "taxonomy_subgroup": "Data Analysis"
    },

    # Data & AI - Data Engineering
    {
        "keywords": [
            "data engineer", "etl developer", "big data engineer", "data warehouse"
        ],
        "taxonomy_group": "Data & AI",
        "taxonomy_subgroup": "Data Engineering"
    },

    # Data & AI - AI/ML
    {
        "keywords": [
            "data scientist", "machine learning", "ml engineer",
            "artificial intelligence", "ai engineer", "deep learning",
            "nlp engineer", "computer vision"
        ],
        "taxonomy_group": "Data & AI",
        "taxonomy_subgroup": "AI / Machine Learning"
    },

    # Infrastructure & Cloud - DevOps
    {
        "keywords": [
            "devops", "site reliability", "sre", "platform engineer",
            "release engineer"
        ],
        "taxonomy_group": "Infrastructure & Cloud",
        "taxonomy_subgroup": "DevOps / SRE"
    },

    # Infrastructure & Cloud - Cloud
    {
        "keywords": [
            "cloud engineer", "cloud architect", "aws engineer",
            "azure engineer", "gcp engineer"
        ],
        "taxonomy_group": "Infrastructure & Cloud",
        "taxonomy_subgroup": "Cloud"
    },

    # Infrastructure & Cloud - System / Network
    {
        "keywords": [
            "system administrator", "systems administrator", "sysadmin",
            "network engineer", "network administrator", "infrastructure engineer"
        ],
        "taxonomy_group": "Infrastructure & Cloud",
        "taxonomy_subgroup": "System / Network"
    },

    # QA & Testing
    {
        "keywords": [
            "tester", "qa engineer", "quality assurance", "test engineer",
            "automation test", "manual test", "software tester"
        ],
        "taxonomy_group": "QA & Testing",
        "taxonomy_subgroup": "Testing / QA"
    },

    # Security
    {
        "keywords": [
            "security analyst", "cybersecurity", "information security",
            "application security", "security engineer", "soc analyst",
            "penetration tester", "pentester"
        ],
        "taxonomy_group": "Security",
        "taxonomy_subgroup": "Cybersecurity"
    },

    # Product / Business / Delivery - BA
    {
        "keywords": [
            "business analyst", "ba", "systems analyst", "system analyst"
        ],
        "taxonomy_group": "Product / Business / Delivery",
        "taxonomy_subgroup": "Business Analysis"
    },

    # Product / Business / Delivery - PM/PO
    {
        "keywords": [
            "project manager", "it project manager",
            "product owner", "product manager", "scrum master",
            "delivery manager"
        ],
        "taxonomy_group": "Product / Business / Delivery",
        "taxonomy_subgroup": "Project / Product Management"
    },

    # Design
    {
        "keywords": [
            "ux designer", "ui designer", "product designer",
            "ux ui", "ui ux", "interaction designer"
        ],
        "taxonomy_group": "Design",
        "taxonomy_subgroup": "UI/UX Design"
    },

    # Support & Operations
    {
        "keywords": [
            "it support", "technical support", "helpdesk",
            "service desk", "desktop support", "application support",
            "database administrator", "dba"
        ],
        "taxonomy_group": "Support & Operations",
        "taxonomy_subgroup": "Support / Operations"
    },
]


def assign_taxonomy(occupation_name: str):
    name = normalize_text(occupation_name)

    for rule in RULES:
        for kw in rule["keywords"]:
            kw_norm = normalize_text(kw)
            if kw_norm in name:
                return pd.Series([rule["taxonomy_group"], rule["taxonomy_subgroup"]])

    return pd.Series(["Other IT", "Unclassified"])

# 6. CHẠY CHÍNH

In [ ]:
def main():
    # Đọc dữ liệu
    df06 = read_table(FILE_06)
    df08 = read_table(FILE_08)

    print("Đã đọc file 06:", df06.shape)
    print("Đã đọc file 08:", df08.shape)

    # Tìm cột quan trọng trong file 06
    occ_id_col_06 = find_column(df06, ["occupation_id", "id", "conceptUri"], required=False)
    occ_name_col_06 = find_column( # tìm một trong các tên cột có thể là tên nghề
        df06,
        ["preferredLabel", "occupation_name", "occupation", "label", "name"]
    )
    group_col_06 = find_column(df06, ["group"], required=False)

    # Tìm cột quan trọng trong file 08
    occ_id_col_08 = find_column(df08, ["occupation_id", "id", "conceptUri"], required=False)
    occ_name_col_08 = find_column(
    df08,
    ["preferredLabel", "occupation_name", "occupation", "occupationLabel", "label", "name"]
)
    skill_count_col_08 = find_column(
    df08,
    ["skill_count", "count_skill", "number_of_skills", "total_skills", "count", "num_skills"]
)

    # Chuẩn hóa tên cột nội bộ cho dễ xử lý
    df06_work = df06.copy()
    df08_work = df08.copy()

    df06_work = df06_work.rename(columns={occ_name_col_06: "occupation_name"})
    df08_work = df08_work.rename(columns={
        occ_name_col_08: "occupation_name",
        skill_count_col_08: "skill_count"
    })

    if occ_id_col_06:
        df06_work = df06_work.rename(columns={occ_id_col_06: "occupation_id"})
    if occ_id_col_08:
        df08_work = df08_work.rename(columns={occ_id_col_08: "occupation_id"})
    if group_col_06:
        df06_work = df06_work.rename(columns={group_col_06: "group"})

    # Tạo key để merge chắc hơn
    df06_work["occupation_name_key"] = df06_work["occupation_name"].apply(normalize_text)
    df08_work["occupation_name_key"] = df08_work["occupation_name"].apply(normalize_text)

    # Merge ưu tiên theo occupation_id nếu cả 2 file đều có
    if "occupation_id" in df06_work.columns and "occupation_id" in df08_work.columns:
        df_merge = df06_work.merge(
            df08_work[["occupation_id", "skill_count"]].drop_duplicates(),
            on="occupation_id",
            how="left"
        )
    else:
        df_merge = df06_work.merge(
            df08_work[["occupation_name_key", "skill_count"]].drop_duplicates(),
            on="occupation_name_key",
            how="left"
        )

    # Điền skill_count thiếu = 0
    df_merge["skill_count"] = df_merge["skill_count"].fillna(0).astype(int)

    # Gán taxonomy
    df_merge[["taxonomy_group", "taxonomy_subgroup"]] = df_merge["occupation_name"].apply(assign_taxonomy)

    # Sắp xếp cho dễ nhìn
    sort_cols = []
    if "group" in df_merge.columns:
        sort_cols.append("group")
    sort_cols += ["taxonomy_group", "taxonomy_subgroup", "skill_count"]

    df_merge = df_merge.sort_values(sort_cols, ascending=[True] * len(sort_cols))

    # Chọn cột đầu ra đẹp hơn
    output_cols = []
    if "occupation_id" in df_merge.columns:
        output_cols.append("occupation_id")
    output_cols.append("occupation_name")
    if "group" in df_merge.columns:
        output_cols.append("group")
    output_cols += [
        "skill_count",
        "taxonomy_group",
        "taxonomy_subgroup"
    ]

    # Nếu muốn giữ cột gốc khác thì thêm vào đây
    remaining_cols = [c for c in df_merge.columns if c not in output_cols and c != "occupation_name_key"]
    final_cols = output_cols + remaining_cols

    df_final = df_merge[final_cols]

    # Xuất file
    output_path = Path(OUTPUT_FILE)
    if output_path.suffix.lower() in [".xlsx", ".xls"]:
        df_final.to_excel(output_path, index=False)
    else:
        df_final.to_csv(output_path, index=False, encoding="utf-8-sig")

    print(f"Đã tạo file: {OUTPUT_FILE}")
    print("Số dòng:", len(df_final))
    print(df_final.head(10))


if __name__ == "__main__":
    main()

Đã đọc file 06: (54, 20)
Đã đọc file 08: (54, 3)
Đã tạo file: 09_occupation_taxonomy_seed.csv
Số dòng: 54
                                        occupation_id  \
12  http://data.europa.eu/esco/occupation/35553663...   
8   http://data.europa.eu/esco/occupation/258e46f9...   
6   http://data.europa.eu/esco/occupation/207d7b18...   
42  http://data.europa.eu/esco/occupation/d3edb8f8...   
5   http://data.europa.eu/esco/occupation/2079755f...   
11  http://data.europa.eu/esco/occupation/349ee6f6...   
39  http://data.europa.eu/esco/occupation/cc867bee...   
10  http://data.europa.eu/esco/occupation/32ac76f6...   
28  http://data.europa.eu/esco/occupation/9e2e6e1e...   
24  http://data.europa.eu/esco/occupation/81480b40...   

                                      occupation_name group  skill_count  \
12                   artificial intelligence engineer  core           89   
8                                      data scientist  core           97   
6                       business intel